In [1]:
from typing import TypedDict, Annotated, List, Dict, Any
import operator
import json
from langgraph.graph import StateGraph, START, END

# =========================================================
# 1. THE STATE & MOCK ENVIRONMENT TOOLS
# =========================================================
class ReActState(TypedDict):
    question: str
    chat_history: Annotated[list, operator.add]
    next_step: str # Control track: "agent", "tools", or "end"

# Simulated enterprise directory structure on your server
MOCK_CODEBASE = {
    "src/auth.ts": "export function login() { return 'token_123'; }\n// Dependency: config.json",
    "src/config.json": "{ 'host': 'api.production.internal', 'port': 8080, 'sec_level': 'high' }"
}

def local_file_reader_tool(filename: str) -> str:
    print(f"🧮 [Tool Execute] Opening and reading: {filename}")
    return MOCK_CODEBASE.get(filename, f"Error: File {filename} not found.")

# =========================================================
# 2. THE REACT AGENT NODE (The Cognitive Engine)
# =========================================================
# We simulate the precise multi-turn text generation an LLM outputs 
# when running a ReAct internal system prompt.

In [2]:
SIMULATION_TURN = 0

def reactive_agent_core(state: ReActState):
    global SIMULATION_TURN
    SIMULATION_TURN += 1
    
    print(f"\n🤖 [Agent Thought Turn #{SIMULATION_TURN}] reviewing workspace history...")
    
    if SIMULATION_TURN == 1:
        # Thought: I need to check auth.ts first to understand the structure
        thought = "Thought: To answer how authentication works, I should read 'src/auth.ts' first to check for dependencies."
        action = {"tool": "local_file_reader_tool", "args": {"filename": "src/auth.ts"}}
        
        print(f"   -> {thought}")
        print(f"   -> Action: Invoke tool with {action['args']}")
        
        return {
            "chat_history": [f"{thought}\nAction: {json.dumps(action)}"],
            "next_step": "tools"
        }
        
    elif SIMULATION_TURN == 2:
        # The agent has inspected the tool observation from turn 1, and sees config.json is needed
        thought = "Thought: The file 'src/auth.ts' depends on 'src/config.json'. I must read that file next to inspect security configurations."
        action = {"tool": "local_file_reader_tool", "args": {"filename": "src/config.json"}}
        
        print(f"   -> {thought}")
        print(f"   -> Action: Invoke tool with {action['args']}")
        
        return {
            "chat_history": [f"{thought}\nAction: {json.dumps(action)}"],
            "next_step": "tools"
        }
        
    else:
        # Final Turn: The agent has all observations and declares completion
        thought = "Thought: I have collected all file parameters. I can now form the final conclusion."
        final_answer = "Final Answer: The system logs in via a token-based handshake protocol configured with 'high' security parameters on port 8080."
        
        print(f"   -> {thought}")
        print(f"   🎉 {final_answer}")
        
        return {
            "chat_history": [f"{thought}\n{final_answer}"],
            "next_step": "end"
        }

In [4]:
# =========================================================
# 3. THE TOOL EXECUTION LAYER NODE
# =========================================================
def tool_execution_executor(state: ReActState):
    # Parse out the last action requested by the agent core
    last_message = state["chat_history"][-1]
    action_json = last_message.split("Action: ")[1]
    action_details = json.loads(action_json)
    
    # Fire the actual local python function mapping the arguments
    filename_arg = action_details["args"]["filename"]
    tool_raw_output = local_file_reader_tool(filename_arg)
    
    observation_string = f"Observation (Tool Output): {tool_raw_output}"
    print(f"   📥 Vectorizing data back to agent workspace...")
    
    return {
        "chat_history": [observation_string],
        "next_step": "agent" # Bounce control straight back to the agent's brain!
    }

In [ ]:
# =========================================================
# 4. CONDITIONAL ROUTING PATHWAYS
# =========================================================
def react_loop_router(state: ReActState):
    return state["next_step"] # Simply matches "tools", "agent", or "end"

# =========================================================
# 5. ASSEMBLING THE GRAPH
# =========================================================
builder = StateGraph(ReActState)

builder.add_node("agent_brain", reactive_agent_core)
builder.add_node("tool_runner", tool_execution_executor)

builder.add_edge(START, "agent_brain")

# The brain evaluates after every single run whether it needs tools or is finished
builder.add_conditional_edges(
    "agent_brain",
    react_loop_router,
    {
        "tools": "tool_runner",
        "agent": "agent_brain",
        "end": END
    }
)

builder.add_edge("tool_runner", "agent_brain")

app = builder.compile()
print("🧠 Autonomous ReAct Loop Architecture Compiled successfully!")

In [ ]:
# =========================================================
# 6. RUN
# =========================================================
prompt = "Analyze our backend workspace security configuration parameters."
print(f"\n🚀 Injecting User Objective: '{prompt}'")

final_state = app.invoke({"question": prompt, "chat_history": [], "next_step": "agent"})